# ESM2 as-standard model

Notebook draft to generate ESM2 embeddings from pylogeny-aware data. 

Tasks
- Import ESM checkpoint
- Import ESM tokenizer
- Import input data (target)
- Preprocess data
- Tokenize data
- Generate embeddings

Downstream tasks
- Run embeddings through classification head pre-trained using lower-level data for token classification
- Assess performance


In [1]:
# Dependancies and libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim

from transformers import AutoModel, AutoTokenizer

import esm

import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

In [2]:
# ESM checkpoints
ESM = ['facebook/esm2_t48_15B_UR50D',
        'facebook/esm2_t36_3B_UR50D',
        'facebook/esm2_t33_650M_UR50D',
        'facebook/esm2_t30_150M_UR50D',
        'facebook/esm2_t12_35M_UR50D',
        'facebook/esm2_t6_8M_UR50D']

In [3]:
# Define checkpoint to be used
checkpoint = ESM[5]

In [4]:
# Create tokenizer and model objects
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# Import target data
df_target= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv")

# Remove extraneous columns
df_target = df_target.iloc[:,0:5]

In [6]:
# Aggreate rows and update format
df_target = df_target.groupby(['Info_protein_id', 'Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

In [7]:
df_target.shape
df_target['label'].str.len().agg(['mean','max'])

mean     410.571429
max     1871.000000
Name: label, dtype: float64

In [8]:
# Create a list of sequences
sequences = df_target['sequence'].tolist()

# Instantiate the tokenizer using the AA sequence lists as input to the tokenizer to create tokenized sequences
inputs = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

# Put the model into evaluation mode
model.eval()

# Generate embeddings for the 
with torch.inference_mode():
    outputs = model(**inputs)

In [9]:
# Verify shape of embedding
outputs.last_hidden_state.size()

torch.Size([21, 1024, 320])

In [10]:
# Freeze the weights in the model
for param in model.parameters():
    param.requires_grad = False

### Generate embeddings from ESM2 for lower level data to train classification head

In [11]:
# Import lower level data
df_lower= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Lower_1763.csv")

# Remove extraneous columns
# df_lower = df_lower.iloc[:,0:5]

# Add mask column where 1 assigned if labelled with pos/neg epitope and -100 if NaN
df_lower['mask'] = df_lower['Class'].isin([-1,1]).astype('int32')
df_lower['mask'] = df_lower['mask'].replace(0, -100)

# Change class column so nan = -100 and -1 = 0. Used later in loss function for classifier training
df_lower['Class'] = df_lower['Class'].fillna(-100).astype('int32')
df_lower['Class'] = df_lower['Class'].replace(-1, 0).astype('int32')



In [12]:
# Aggregate columns for wide format
df_lower = df_lower.groupby(['Info_protein_id','Info_group','Info_split'], as_index=False).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list),
    mask=('mask', list))

df_lower

,Info_protein_id,Info_group,Info_split,sequence,label,position,mask
0,A1KFU9.1,215.0,split_01_20,MAENSNIDDIKAPLLAALGAADLALATVNELITNLRERAEETRTDT...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,A43589,252.0,split_05_20,MLGNAPSVVPNTTLGMHCGSFGSAPSNGWLKLGLVEFGGVAKLNAE...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,AAA21416.1,10.0,split_05_20,MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLRE...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
3,AAA21417.1,239.0,split_02_20,MLDVNFFDELRIGLATAEDIRQWSYGEVKKPETINYRTLKPEKDGL...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
4,AAA25359.1,146.0,split_02_20,MTDVSRKIRAWGRRLMIGTAAAVVLPGLVGLAGGAATAGAFSRPGL...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
...,...,...,...,...,...,...,...
334,YP_178023.1,276.0,split_04_20,MTEQQWNFAGIEAAASAIQGNVTSIHSLLDEGKQSLTKLAAAWGGS...,"[-100, -100, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
335,YP_976577.1,46.0,split_05_20,MAKTIAYDEEARRGLERGLNALADAVKVTLGPKGRNVVLEKKWGAP...,"[-100, -100, -100, -100, -100, -100, 1, 1, 1, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, 1, 1, 1, ..."
336,ZP_03425930.1,45.0,split_02_20,MAEELHAAAGSFASVTTGLAGDAWHGPASLAMTRAASPYVGWLNTA...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
337,ZP_03432777.1,17.0,split_02_20,MTDRVSVGNLRIARVLYDFVNNEALPGTDIDPDSFWAGVDKVVADL...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."


In [13]:
# # Create a class column that checks whether the sequence contains a positive or negative epitope and apply a class column for the stratified grouped k fold
# df_lower["class"] = df_lower["label"].apply(lambda x: 1 if 1 in x else -1)

# # Check max length of sequences
# df_lower['label'].str.len().agg(['mean','max'])

Df contains sequences with length > ESM max input length (1024), sliding window will need to be applied once draft complete - same applies for target data

In [14]:
# # Info_group as the grouping variable and Class  / label as the stratification variable.
# X = df_lower.index
# y = df_lower['class']
# groups = df_lower['Info_group']

# # Instantiate GroupShuffleSplit instance to create grouped train/test splits, use 20% of the data for a hold out/test set
# gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# # Split the data into train/test splits, create indices to be used to assign train/test labels to the df
# train_cv_idx, test_idx = next(gss.split(X, y, groups))

# # Create a train/test column in the dataframe and set the values of the test rows to train or test
# df_lower.loc[test_idx, 'train_test'] = 'test'
# df_lower.loc[train_cv_idx, 'train_test'] = 'train'

# # Update X, y and groups with the remaining train_cv_idx indices to use in Statified Grouped k fold
# X_train, y_train, groups_train = X[train_cv_idx], y[train_cv_idx], groups[train_cv_idx]


In [15]:
# # Split into different folds ensuring stratification accross groups
# sgkf = StratifiedGroupKFold(n_splits=5)

# X_train_df = pd.DataFrame(index=range(0,len(df_lower)))
                                     
# for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
#     X_train_df.loc[train_idx, f'training_split {fold+1}'] = 1
#     X_train_df.loc[val_idx, f'training_split {fold+1}']= 2

# df_lower = df_lower.merge(X_train_df, left_index=True, right_index=True)

In [16]:
# Create train/val/test splits 
# df_lower_cv = df_lower[df_lower['train_test'] == 'train'].reset_index()
# df_lower_test = df_lower[df_lower['train_test'] == 'test'].reset_index()


# Create validation splits from the info split column
cv_1_train = df_lower[df_lower['Info_split'] != 'split_01_20'].reset_index()
cv_1_val = df_lower[df_lower['Info_split'] == 'split_01_20'].reset_index()

cv_2_train = df_lower[df_lower['Info_split'] != 'split_02_20'].reset_index()
cv_2_val = df_lower[df_lower['Info_split'] == 'split_02_20'].reset_index()

cv_3_train = df_lower[df_lower['Info_split'] != 'split_03_20'].reset_index()
cv_3_val = df_lower[df_lower['Info_split'] == 'split_03_20'].reset_index()

cv_4_train = df_lower[df_lower['Info_split'] != 'split_04_20'].reset_index()
cv_4_val = df_lower[df_lower['Info_split'] == 'split_04_20'].reset_index()

cv_5_train = df_lower[df_lower['Info_split'] != 'split_05_20'].reset_index()
cv_5_val = df_lower[df_lower['Info_split'] == 'split_05_20'].reset_index()


In [17]:
# Create custom dataset class
class SequenceDataset(torch.utils.data.Dataset):
    
    def __init__(self, df):
        self.protein_id = df['Info_protein_id']
        self.sequence = df['sequence']
        self.position = df['position']
        self.labels = df['label']
        self.label_mask = df['mask']
        
    def __len__(self):
        return len(self.sequence)
        
    def __getitem__(self, idx):
        return {
            'protein_id': self.protein_id[idx],
            'sequence': self.sequence[idx],
            'position': self.position[idx],
            'label': self.labels[idx],
            'mask': self.label_mask[idx]
        }

Generate embeddings for each of the lower level sequences 

In [18]:
# Create a custom collator to maintain length of items within the batch
def collate_fn(batch):
    return {
        'protein_id': [x['protein_id'] for x in batch],
        'sequence': [x['sequence'] for x in batch],
        'position': [x['position'] for x in batch],
        'label': [x['label'] for x in batch],
        'mask': [x['mask'] for x in batch],
    }

# Function to create a DataLoader instance using the cv dataset and the custom collate function
def batch_create(dataset, tokenizer=tokenizer):    
    # create a DataLoader instance using the cv dataset and the custom collate function
    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=True,
        collate_fn=collate_fn
    )
    
    emb_output_list = []
    
    # Loop through each batch of tensors and apply tokenisation to each sequence, using max length of 1024 (will require windowing and averaging in later versions)
    for batch in loader:
    
        inputs = tokenizer(
            batch['sequence'],
            padding=True,
            truncation=True,
            max_length=1024,
            return_tensors='pt'
        )
    
        # Freeze model weights and calculate embeddings for the tokenized embeddings (remove start and end CLS/EOS tokens)
        with torch.no_grad():
            embeddings = model(**inputs).last_hidden_state[:,1:-1,:]
    
        # Convert label and mask to tensors 
        label = [torch.tensor(item) for item in batch['label']]
        mask = [torch.tensor(item) for item in batch['mask']]
        
        # Pad each label and mask sequence to the size of the largest embedding within the batch
        label = pad_sequence(label, padding_value=-100, padding_side='right')[0:embeddings.size()[1]]
        mask = pad_sequence(mask, padding_value=-100, padding_side='right')[0:embeddings.size()[1]]
    
        # print('embeddings', embeddings.size())
        # print('label', label.size())
        # print('mask', mask.size())
        
        # Append the embeddings, label and masks per batch to an output list 
        emb_output_list.append({'embeddings': embeddings,
                         'label': label, 
                         'mask': mask})
    
    return emb_output_list

In [19]:
# Create datasets
train_datasets = {
    1:SequenceDataset(cv_1_train),
    2:SequenceDataset(cv_2_train), 
    3:SequenceDataset(cv_3_train),
    4:SequenceDataset(cv_4_train),
    5:SequenceDataset(cv_5_train)    
}

val_datasets = {
    1:SequenceDataset(cv_1_val),
    2:SequenceDataset(cv_2_val), 
    3:SequenceDataset(cv_3_val),
    4:SequenceDataset(cv_4_val),
    5:SequenceDataset(cv_5_val)    
}

# Create train batches
train_loaded = {}

for key, value in train_datasets.items():
    batched = batch_create(value, tokenizer=tokenizer)
    train_loaded[key] = batched

train_loaded.items()

# Create val batches
val_loaded = {}

for key, value in val_datasets.items():
    batched = batch_create(value, tokenizer=tokenizer)
    val_loaded[key] = batched

### Train classification head on lower level data

Tasks
- Instantiate classifier for token classification
- Train
- Test

In [23]:
# Create simple 2 layer neural network in PyTorch to return logits per residue representing each class
class PerResidueClassifier(nn.Module):
    def __init__(self, in_features=320, hidden_size=128, out_features=2):
        super().__init__()
        
        self.linear1 = nn.Linear(in_features, hidden_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, out_features)
        
    def forward(self, embeddings):
        x = self.linear1(embeddings)
        x = self.relu(x)
        logits = self.linear2(x)
        
        return logits

In [24]:
# Instantiate classifier
clf = PerResidueClassifier()

# # Create classifer out
# clf_outputs = []

# for emb_batch in emb_output_list:
#     outputs = clf(emb_batch['embeddings'])
#     clf_outputs.append({'clf_logits': outputs,
#                         'label': emb_batch['label'],
#                         'mask': emb_batch['mask']}
#                       )    

In [54]:
# Instantiate cross-entropy loss function, ignores positions with mask -100
loss_fcn = nn.CrossEntropyLoss(ignore_index=-100)

# n_folds = len(train_loaded.items())

training_loss_dict = {}
val_loss_dict = {}

epochs = 20

# Training loop 
for key, fold in train_loaded.items():
    
    training_loss_dict[key] = []
    val_loss_dict[key] = []
    
    # Classifier
    clf = PerResidueClassifier()

    # Adam optimiser
    optimiser = optim.Adam(clf.parameters(), lr=0.001)
    
    for epoch_n, epoch in enumerate(range(0, epochs)):
        
        clf.train()
        running_loss = 0
        
        for  batch_n, batch in enumerate(fold):
            # print(key, epoch_n, fold_n)
            inputs = batch['embeddings']
            labels = batch['label']
        
            # Zero model gradients per batch
            optimiser.zero_grad()
        
            # Caluclate logits by passing embeddings through the classifier
            outputs = clf(inputs)
        
            # Compute loss and gradients        
            loss = loss_fcn(outputs.reshape(-1, 2), labels.reshape(-1))
    
            # Calculate the gradients through the network
            loss.backward()
    
            # Adjust learning weights
            optimiser.step()

            # Add the loss to a running counter
            running_loss += loss.item()

        # Calculate loss per epoch
        epoch_loss = running_loss/len(fold)

        training_loss_dict[key].append(epoch_loss)
        
        # Put the classifier in eval mode
        clf.eval()

        running_val_loss = 0

        for batch in val_loaded[key]:
            inputs = batch['embeddings']
            labels = batch['label']
            
            with torch.no_grad():
                val_outputs = clf(inputs)

            val_loss = loss_fcn(val_outputs.reshape(-1, 2), labels.reshape(-1))

            running_val_loss += val_loss.item()

        epoch_val_loss = running_val_loss/len(val_loaded[key])

        val_loss_dict[key].append(epoch_val_loss)

print(f'Training loss: {training_loss_dict.items()}')
print(f'Validation loss: {val_loss_dict.items()}')
        

Training loss: dict_items([(1, [0.7071827761828899, 0.682599164545536, 0.6786330714821815, 0.6728598661720753, 0.6662665344774723, 0.6595560386776924, 0.6526848450303078, 0.645401768386364, 0.6382101289927959, 0.6307016983628273, 0.6233404502272606, 0.6159015446901321, 0.6084908060729504, 0.6009873859584332, 0.5935406628996134, 0.5859523713588715, 0.5785827897489071, 0.5710958726704121, 0.5637463591992855, 0.5564293712377548]), (2, [0.6879789663685693, 0.6771681242518954, 0.6715981430477567, 0.6658560269408755, 0.6596351398362054, 0.6527321206198798, 0.6455587910281287, 0.6381120847331153, 0.6305891937679715, 0.6231691737969717, 0.6154175831211938, 0.6079167657428317, 0.6000703109635247, 0.5923244009415308, 0.5844042946894964, 0.5765255441268285, 0.568632142411338, 0.5605607910288705, 0.5526190350453059, 0.5446203069554435]), (3, [0.6941742125679465, 0.6805667701889487, 0.6739687814432032, 0.6675567802260903, 0.6619165715049294, 0.6548979457686929, 0.6479150863254771, 0.640166647293988

In [55]:
# Average training loss per epoch
for key, value in training_loss_dict.items():
    trainiprint(key)
    print(value)





1
[0.7071827761828899, 0.682599164545536, 0.6786330714821815, 0.6728598661720753, 0.6662665344774723, 0.6595560386776924, 0.6526848450303078, 0.645401768386364, 0.6382101289927959, 0.6307016983628273, 0.6233404502272606, 0.6159015446901321, 0.6084908060729504, 0.6009873859584332, 0.5935406628996134, 0.5859523713588715, 0.5785827897489071, 0.5710958726704121, 0.5637463591992855, 0.5564293712377548]
2
[0.6879789663685693, 0.6771681242518954, 0.6715981430477567, 0.6658560269408755, 0.6596351398362054, 0.6527321206198798, 0.6455587910281287, 0.6381120847331153, 0.6305891937679715, 0.6231691737969717, 0.6154175831211938, 0.6079167657428317, 0.6000703109635247, 0.5923244009415308, 0.5844042946894964, 0.5765255441268285, 0.568632142411338, 0.5605607910288705, 0.5526190350453059, 0.5446203069554435]
3
[0.6941742125679465, 0.6805667701889487, 0.6739687814432032, 0.6675567802260903, 0.6619165715049294, 0.6548979457686929, 0.6479150863254771, 0.6401666472939884, 0.6321612280957839, 0.623947995550